# Class 3 — LLMOps: evaluating and promoting prompts

Class 2 trained a model, registered it, promoted a version with `@champion`, and the API
served whatever the alias pointed at.

This notebook does the same thing to **prompts**.

| | Class 2 | Class 3 |
|---|---|---|
| The thing you version | a trained model | a **Prompt Mode** |
| How you compare them | a held-out test set | the **Evaluation Set** |
| How you ship one | move `@champion` | move `@champion` |
| What serves it | the prediction API | BonsAI, at http://localhost:3000 |

Everything here imports from `src/`, the same modules the running service uses. What you
measure in this notebook is exactly what customers get.

In [ ]:
# The environment is already built — nothing to install.
# See docker/Dockerfile.jupyter and docker/requirements-notebook.txt.
import logging

# The OpenAI SDK logs every HTTP request. Useful when debugging, noise in a lesson —
# turn it back up if you want to watch the retries happen.
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

from src import llm_client, prompt_modes, evaluation_set
from src.evaluate_prompts import score_mode, RateLimiter, looks_like_refusal

import mlflow
import mlflow.genai
import pandas as pd

config = llm_client.describe()
for key, value in config.items():
    print(f"{key:>20}: {value}")

if not config["api_key_configured"]:
    print("\nNo GEMINI_API_KEY. Put it in docker/.env and restart the stack.")

## 1. The four Prompt Modes

Four different ways of instructing the same model. They live in
[`src/prompt_modes.py`](../src/prompt_modes.py), imported by both this notebook and the
BonsAI service — so there is exactly one definition of each.

In [ ]:
pd.DataFrame([
    {"mode": name, "description": mode["description"], "characters": len(mode["template"])}
    for name, mode in prompt_modes.PROMPT_MODES.items()
])

In [ ]:
# What one actually looks like once the customer's question is filled in.
print(prompt_modes.render("diagnostic", "My bonsai leaves are turning yellow"))

## 2. Ask it something

One call, so you can see a real answer before we start measuring them.

In [ ]:
client = llm_client.build_client()

def ask(question, mode="basic"):
    """Ask BonsAI one question, and fail in a way you can read."""
    try:
        return client.chat.completions.create(
            model=llm_client.get_model(),
            messages=[{"role": "user", "content": prompt_modes.render(mode, question)}],
            max_tokens=llm_client.get_max_tokens(),
        )
    except Exception as exc:
        # A shared free model goes through spells of 503 "high demand", and quota runs out
        # as 429. Neither is a bug in your prompt, and neither should end the lesson in a
        # traceback.
        print(f"The provider refused the call: {type(exc).__name__}")
        print(str(exc)[:200])
        return None

response = ask("How often should I water my Juniper bonsai?")

if response:
    choice = response.choices[0]
    print(choice.message.content)
    print(f"\nfinish_reason: {choice.finish_reason} | tokens: {response.usage.total_tokens}")

### Watch `finish_reason`

If it says `length` instead of `stop`, the answer was cut off — and nothing raised an
error. Gemini spends part of the token budget *reasoning* before it writes anything, so a
budget that looks generous can still run out mid-sentence.

That matters more than it sounds: a truncated answer still gets scored. It does not crash
the pipeline, it just quietly lowers a number. Try `max_tokens=300` above and see.

## 3. The Evaluation Set

The fixed bar. Real questions with reference answers, **and** questions BonsAI must
refuse — a specialist that cheerfully answers about tomatoes is broken, however fluent it
sounds.

It is not training data. Nothing is fitted to it. It exists so two Prompt Modes can be
compared on identical input, which is why it must stay fixed: change the ruler and
yesterday's scores stop meaning anything.

In [ ]:
pd.DataFrame([
    {"kind": case["kind"], "query": case["query"],
     "must_mention": ", ".join(case.get("must_mention", []))}
    for case in evaluation_set.all_cases()
])

## 4. Score one mode

Now the mechanics, on a sample rather than the whole set — the free tier allows **5
requests per minute**, so the full run takes about eight minutes. We come back to that.

`sample()` keeps both kinds deliberately. A subset of on-topic questions only would leave
`refusal_accuracy` undefined, and a mode that answers everything would stop looking wrong.

In [ ]:
demo_cases = evaluation_set.sample(n_on_topic=2, n_off_topic=1)
limiter = RateLimiter(requests_per_minute=5)

metrics = score_mode(
    client=client,
    model=llm_client.get_model(),
    max_tokens=llm_client.get_max_tokens(),
    mode_name="basic",
    template=prompt_modes.PROMPT_MODES["basic"]["template"],
    limiter=limiter,
    cases=demo_cases,
)

pd.Series(metrics)

### Read `failed_calls` before you read the score

A call rejected with `429` still produces a score — of zero. So a low score can mean "this
prompt is bad" or "you ran out of quota", and the number looks identical either way.

This is the failure mode to remember from this class: **LLM evaluation fails quietly**. It
does not crash, it returns a plausible number. `failed_calls` and `truncated_responses`
sit next to every score so you can tell the difference.

## 5. Compare all four

This is the real run: four modes over ten cases, forty calls, paced at five a minute. It
takes about eight minutes, and it is the same code the CI pipeline runs —
[`src/evaluate_prompts.py`](../src/evaluate_prompts.py).

Run it from a terminal so you keep the notebook usable:

```bash
docker compose exec bonsai python -m src.evaluate_prompts --promote
```

Or run it here and talk through the metrics while it works.

In [ ]:
# Uncomment to run the full comparison from the notebook (~8 minutes).
# !cd /home/jovyan && python -m src.evaluate_prompts --promote

## 6. Look at the results

Open MLflow at **http://localhost:5001** and compare the runs side by side, exactly as you
compared model runs in class 2.

In [ ]:
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("Bonsai-Care-Prompt-Engineering")

runs = mlflow.search_runs(order_by=["start_time DESC"], max_results=10)

columns = [c for c in [
    "tags.mlflow.runName", "metrics.overall_score", "metrics.key_point_coverage",
    "metrics.refusal_accuracy", "metrics.rougeL", "metrics.failed_calls",
] if c in runs.columns]

runs[columns].dropna(subset=["metrics.overall_score"]) if columns else runs.head()

## 7. Who is the champion?

Every mode was registered as a *version* of one prompt, `bonsai-care`. The winner carries
the `@champion` alias — one pointer, one live version, exactly like the model registry in
class 2.

In [ ]:
champion = mlflow.genai.load_prompt(f"prompts:/{prompt_modes.PROMPT_NAME}@champion")

print(f"{prompt_modes.PROMPT_NAME} @champion -> version {champion.version}")
print(f"mode: {(champion.tags or {}).get('mode')}")
print(f"variables: {champion.variables}")
print()
print(champion.template[:400])

## 8. Moving the alias is the deployment

BonsAI caches the prompt after loading it, so it keeps serving the old one until told
otherwise. Ask it what it is serving, reload, and ask again.

Nothing is rebuilt. Nothing is redeployed. The same move as class 2, where promoting a
model version changed what the API answered without touching the API.

In [ ]:
import requests

def serving():
    info = requests.get("http://bonsai:3000/health", timeout=15).json()["model_info"]
    return f"{info['status']} | mode={info['mode']} | version={info['version']}"

print("before:", serving())
requests.post("http://bonsai:3000/prompt/reload", timeout=60)
print("after: ", serving())

In [ ]:
# And the champion answering a customer.
answer = requests.post(
    "http://bonsai:3000/chat",
    json={"query": "My Juniper bonsai has brown tips. What should I do?"},
    timeout=120,
).json()

print(answer["response"])

## What to take away

- A prompt is a **versioned artifact**, not a string buried in the app
- An evaluation is only worth what its **set** is worth, and the set must stay fixed
- Scoring one thing and serving another proves nothing — which is why the modes live in
  one file that both sides import
- **Quota and token limits corrupt results silently**; measure and report them alongside
  every score
- Promotion is moving an alias, and rollback is moving it back

## Try it

1. Add a fifth Prompt Mode to `src/prompt_modes.py` and rerun the comparison
2. Add a question to the Evaluation Set that the current champion gets wrong
3. Set `max_tokens=300` and watch `truncated_responses` rise while scores stay plausible